In [25]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float", lambda x: f"{x:,.2f}")

DB_PATH = r"../customer_data/vault_chain_flow.db"
TABLE = "customer_Transactions"

In [26]:
con = sqlite3.connect(DB_PATH)
cursor = con.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tables in database:", cursor.fetchall())

Tables in database: [('Customer_Transactions',)]


In [27]:
df = pd.read_sql_query(f"SELECT * FROM Customer_Transactions", con)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns.")

Loaded 580,962 rows and 14 columns.


In [28]:
print("shape (rows, columns):", df.shape)
print("Total rows:", f"{df.shape[0]:,}")
print("Total columns:", df.shape[1])

shape (rows, columns): (580962, 14)
Total rows: 580,962
Total columns: 14


In [29]:
col_info = pd.DataFrame({
    "column": df.columns,
    "Dtype": df.dtypes.astype(str).values,
    "Non-Null": df.notna().sum().values,
    "Null": df.isna().sum().values,
    "Null %": (df.isna().mean() * 100).round(2).values,
    "unique": df.nunique(dropna=True).values,
})
print(col_info.to_string(index=False))

             column   Dtype  Non-Null  Null  Null %  unique
      TransactionID     str    580962     0    0.00  580962
         CustomerID     str    580962     0    0.00    4997
    TransactionDate     str    580962     0    0.00     639
    TransactionTime     str    580962     0    0.00   71454
    TransactionType     str    580962     0    0.00       9
  TransactionAmount float64    580962     0    0.00  306886
TransactionCurrency     str    569116 11846    2.04       6
          AssetType     str    566487 14475    2.49       5
        CustomerDOB     str    580962     0    0.00    4393
         CustGender     str    580962     0    0.00       2
       CustLocation     str    563766 17196    2.96      23
 CustAccountBalance float64    580962     0    0.00  407472
     WalletActivity   int64    580962     0    0.00       2
  TransactionStatus     str    580962     0    0.00       5


Missing Values Per Column

Quick one-liner that returns the total count of NAN values fro every column in df
Useful as a fast inline check of data completeness especially right after parsing dates (eg, pd.to_datetime silently convert unparseable values to Nat, which shows up here as missing)

Compare against the earlier missing DataFrame to confirm nothing new was introduced by recent transactions



In [30]:
df.isna().sum()

TransactionID              0
CustomerID                 0
TransactionDate            0
TransactionTime            0
TransactionType            0
TransactionAmount          0
TransactionCurrency    11846
AssetType              14475
CustomerDOB                0
CustGender                 0
CustLocation           17196
CustAccountBalance         0
WalletActivity             0
TransactionStatus          0
dtype: int64

Imputing Missing TransactionCurrency with a Rule-Based Cascade

TransactionCurrency is a critical field - every transaction should have a currency for aggregation. FX conversion and reconcillation. Rather than dropping rows or applying a naive global fill (eg, model) we apply a prioritized cascade of 5 rules,  ordered from most-specific and reliable to most-generic fallback. Each rule targets only the rows still missing a currency. so earlier, higher-confidence rules always take precedence.

A shared lookup table maps each ASSetType to it natural settlement currency: BTC->BTC, ETH-->ETH, USDT--> USDT, and stablecoin --> USDT (since stablecoins settle in USDT)

RULE 1 AssetType drives currency

Logic: If TransactionCurrency is missing and the row has a recognized AssetType , infer the currency directly from the asset.

Why first? Asset type is the strongest signal --a BYC transaction is almost certainly denominated in BTC.
The mask ensures we only touch rows that are both missing currency and have a known asset
Print how many rows this rule filled

RULE 2: Crypto transaction type -. currency from asset

Logic: Crypto Buy/sell transactions with a known asset inherit that asset"s currency -- even if Rule 1 didnt already catch them
Why here Reinforces the crypto--> asset currency assumption using transactionType as an additional qualifier
Requires Assettype to be present so the lookup producesa valid value.

RULE 3: Wallet - only movement default to GBP
Logic: Rows flagged as wallet activity with no currency are interanl wallet movements which defaultbto the home currency (GBP)
why GBP? internal wallet activity doesnt cross borbers so it settle in the platform"s home currency 

RULE 4: 
Cross-border defaults to USD
Logic: Cross-Border payement transactions without a currency default to USD the de facto standard for international settlement

RULE 5: Everthing else --> GBP (home currency)
Logic: Catch all fallback-- any remaining rows missing a currency are filled with GBP.
Why last? This is the lowest-confidence rule and should only apply after allspecific rules have had a chance to fire. it gurantees zero remaining nulls

Final Verification
confirms the casacade fully resolved the column (expected: 0 remaining nulls)

KEY DESIGN PRINCIPLES
principle       why it matters
Ordered specificity     Rules run from most-specific (asset -> currency) to most-genericfallbacl --> GBP so high
                        confidence inference always win.
Null-guarded masks       Every rule only applies where the currency is still missingso no existing values                 ever                overwrtten

observability:  each rule point how many rows it filled--you can audit which rules heavy liftting and detect unexpected shift in data ditribution

idempotent and reproducible  Rules are deterministic re-running the block on an already-filled column is ano-op

In [31]:
# Rule 1 AssetType drives currency
asset_to_currency = {
    "BTC": "BTC",
    "ETH": "ETH",
    "USDT": "USDT",
    "Stablecoin": "USDT",
}
mask = df["TransactionCurrency"].isna() & df["AssetType"].isin(asset_to_currency)
df.loc[mask, "TransactionCurrency"] = df.loc[mask, "AssetType"].map(asset_to_currency)
print(f"Rule 1 filled: {mask.sum():,}")

# Rule 2 - Crypto transaction type -> currency from asset (if asset present)
mask = (df["TransactionCurrency"].isna()
        & df["TransactionType"].isin(["Crypto Buy", "Crypto Sell"])
        & df["AssetType"].notna())
df.loc[mask, "transactionCurrency"] = df.loc[mask, "AssetType"].map(asset_to_currency)
print(f"Rule 2 filled: {mask.sum():,}")

# Rule 3 - wallet - only movements default to GBP
mask = df["TransactionCurrency"].isna() & (df["WalletActivity"] == 1)
df.loc[mask, "TransactionCurrency"] = "GBP"
print(f"Rule 3 filled: {mask.sum():,}")

# Rule 4-- cross - Border defaults to USD
mask = df["TransactionCurrency"].isna() & (df["TransactionType"] == "Cross-Border payment")
df.loc[mask, "TransactionCurrency"] = "USD"
print(f"Rule 4 filled: {mask.sum():,}")

# Rule 5 - Everything else-> GBP (home currency)
mask = df["TransactionCurrency"].isna()
df.loc[mask, "TransactionCurrency"] = "GBP"
print(f"Rule 5 filled: {mask.sum():,}")

print("\nRemaining TransactionCurrency nulls:", df["TransactionCurrency"].isna().sum())

Rule 1 filled: 3,217
Rule 2 filled: 0
Rule 3 filled: 1,835
Rule 4 filled: 0
Rule 5 filled: 6,794

Remaining TransactionCurrency nulls: 0
